# Análisis de perfiles longitudinales, χ , ksn, ScaledRelief

Este notebook analiza los cauces asociados a los 18 outlets definidos
para el área de estudio mediante la biblioteca TopoAnalysis.

## Datos de entrada

- Modelo digital de elevación original para obtener las elevaciones.
- Grilla de dirección de flujo.
- Grilla de área de drenaje.
- Coordenadas de los outlets y nombres de los ríos.

Los datos espaciales deben compartir el sistema de coordenadas,
la resolución y la extensión.

## Procedimiento

1. Definir directorios, parámetros y outlets.
2. Cargar y comprobar las grillas topográficas.
3. Extraer el cauce principal de cada río.
4. Construir los perfiles de distancia–elevación.
5. Calcular χ y representar los perfiles χ–elevación.
6. Calcular el índice de pendiente normalizada, ksn.
7. Exportar las figuras y las tablas de resultados.

La distancia y χ tendrán su origen en el outlet de cada río.
El índice de concavidad de referencia y el área mínima de drenaje
se definirán explícitamente antes del cálculo.

## Resultados

- Perfiles longitudinales de distancia–elevación.
- Perfiles de χ–elevación.
- Valores de ksn y sus gráficos.
- Tablas con coordenadas, distancia, elevación, área de drenaje, χ y ksn.

In [ ]:
# ── Directorios y rutas de archivos ───────────────────────────────────────
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress
from TopoAnalysis import dem as d
print("Bibliotecas importadas correctamentee.")

## 1. Directorios y parámetros del análisis

Se definen las carpetas de entrada y salida y los parámetros utilizados
para extraer los cauces y calcular χ y ksn.

- **Área de referencia (`A0`):** normalización para calcular χ.
- **Área mínima (`AREA_THRESHOLD`):** área de drenaje mínima para incluir
  un punto en la red analizada.
- **Concavidad de referencia (`THETA_REF`):** exponente utilizado en
  los cálculos de χ y ksn. Se adopta inicialmente 0,45, sujeto a revisión.

Las áreas se expresan en metros cuadrados. Con esta normalización,
χ se expresa en metros.

Las figuras y tablas se guardarán en una carpeta propia de este análisis.

In [ ]:
# ── Directorios de trabajo ───
BASE_DIR = Path(
    r"C:\datos\Memoria\Codigos Stanford"
)

# Carpeta con los productos del procesamiento topográfico anterior
DATA_DIR = BASE_DIR / "Productos"
# Nombre de este análisis
NB_NAME = "perfiles_chi_ksn"
# Carpetas de salida
OUTPUT_DIR = DATA_DIR / NB_NAME
FIG_DIR = OUTPUT_DIR / "figuras"
RES_DIR = OUTPUT_DIR / "tablas"

if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"No se encontró la carpeta de entrada: {DATA_DIR}"
    )

for folder in (FIG_DIR, RES_DIR):
    folder.mkdir(parents=True, exist_ok=True)


# ── Parámetros del análisis ──────────────────────────────────────────────
A0 = 1e6                 # m² — área de referencia para χ (1 km²)
AREA_THRESHOLD = 1e6     # m² — área mínima de drenaje (1 km²)
THETA_REF = 0.45         # Concavidad de referencia inicial; adimensional

# ── Configuración de figuras ─────────────────────────────────────────────
plt.rcParams.update({
    "figure.figsize": (9, 6),
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

# ── Resumen ──────────────────────────────────────────────────────────────
print(f"Datos de entrada: {DATA_DIR}")
print(f"Figuras:          {FIG_DIR}")
print(f"Tablas:           {RES_DIR}")
print()
print(f"Área de referencia:     {A0 / 1e6:g} km²")
print(f"Área mínima de drenaje: {AREA_THRESHOLD / 1e6:g} km²")
print(f"Concavidad inicial:    {THETA_REF}")

# Outlets
Se definen los nombres y las coordenadas de los 18 outlets del análisis.

Las coordenadas corresponden a Este (X) y Norte (Y), en metros,
en el sistema UTM del proyecto. Deben coincidir con el sistema
de coordenadas de las grillas topográficas.

Cada outlet será el origen de la distancia y de χ para su río.

In [ ]:
sample_outlets = {
    "Rio Huenchullami":       (769308, 6089892),
    "Estero Cañete":          (740807, 6092244),
    "Estero Quebrada Honda":  (728736, 6079032),
    "Estero Malhueco":        (729540, 6072948),
    "Rio Pinotalca":          (719340, 6060852),
    "Rio Reloca":             (717258, 6051540),
    "Estero de Chanco":       (720425, 6039864),
    "Rio Rahue":              (724668, 6037584),
    "Rio Curanilahue":        (719400, 6034812),
    "Rio Curanipe":           (713916, 6030936),
    "Rio Chovellen":          (714756, 6021096),
    "Tregalemu":              (703485, 6018819),
    "Limite Maule-Ñuble":     (702277, 6012996),
    "Estero Buchupureo":      (705000, 6006276),
    "Cobquecura":             (702961, 6000696),
    "Rio Taucu":              (701496, 5994768),
    "Estero Colmuyado":       (699396, 5979828),
    "Estero Mela":            (697332, 5976096),
}

# Asignar un color a cada río para reutilizarlo en las figuras
cmap = plt.get_cmap("tab20")
colors = {
    name: cmap(i)
    for i, name in enumerate(sample_outlets)
}

# Tabla de consulta
outlets_df = pd.DataFrame(
    [
        {"rio": name, "x_utm": x, "y_utm": y}
        for name, (x, y) in sample_outlets.items()
    ]
)

print(f"Outlets definidos: {len(sample_outlets)}")
display(outlets_df)

## 3. Carga de los datos topográficos

Se cargan las grillas generadas en el notebook anterior:

- **Dirección de flujo D8:** define las conexiones entre las celdas.
- **Área de drenaje:** se utiliza para seleccionar los cauces y calcular χ y ksn.
- **Elevación original:** proporciona las cotas de los perfiles longitudinales.

Los nombres de los archivos conservan el prefijo `rapel` del procesamiento
anterior. El nombre del archivo no determina su resolución espacial.

Se comprueba que las tres matrices tengan las mismas dimensiones.
La coincidencia del sistema de coordenadas y de la alineación espacial
también debe verificarse antes de extraer los perfiles.

In [ ]:
# ── Rutas de entrada ─────────────────────────────────────────────────────
AREA_PATH = DATA_DIR / "rapel_area_utm30m"
FD_PATH = DATA_DIR / "rapel_fd_utm30m"

# DEM original utilizado en el notebook anterior, con ruta para Windows
DEM_PATH = BASE_DIR.parent / "TDM_cut_UTM18.tif"

# Comprobar que existan las entradas
for path in (AREA_PATH, FD_PATH, DEM_PATH):
    if not path.exists():
        raise FileNotFoundError(f"No se encontró: {path}")


# ── Cargar las grillas con TopoAnalysis ───────────────────────────────────
area = d.Area.load(str(AREA_PATH))
fd = d.FlowDirectionD8.load(str(FD_PATH))
elevation = d.Elevation.load(str(DEM_PATH))


# ── Acceder a las matrices ───────────────────────────────────────────────
area_grid = area._griddata
fd_grid = fd._griddata
elev_grid = elevation._griddata


# ── Comprobar dimensiones ────────────────────────────────────────────────
if not (area_grid.shape == fd_grid.shape == elev_grid.shape):
    raise ValueError(
        "Las grillas tienen dimensiones diferentes:\n"
        f"Área: {area_grid.shape}\n"
        f"Dirección de flujo: {fd_grid.shape}\n"
        f"Elevación: {elev_grid.shape}"
    )

print("Grillas cargadas correctamente.")
print(f"Dimensiones: {elev_grid.shape[0]} filas × "
      f"{elev_grid.shape[1]} columnas")
print(f"DEM de elevación: {DEM_PATH.name}")
print("Siguiente paso: comprobar la alineación espacial y los outlets.")

## 4. Comprobación espacial

Verificamos que las grillas compartan sistema de coordenadas,
resolución y alineación.

In [ ]:
import rasterio

# ── Leer la información espacial ─────────────────────────────────────────
grid_info = {}

for name, path in {
    "Elevación": DEM_PATH,
    "Área": AREA_PATH,
    "Dirección": FD_PATH,
}.items():
    with rasterio.open(path) as src:
        grid_info[name] = {
            "crs": src.crs,
            "transform": src.transform,
            "shape": src.shape,
            "res": src.res,
        }

reference = grid_info["Elevación"]

if reference["crs"] is None:
    raise ValueError("El DEM no tiene un sistema de coordenadas definido.")

if not reference["crs"].is_projected:
    raise ValueError("El DEM debe estar en coordenadas proyectadas.")

for name, info in grid_info.items():
    if info["crs"] != reference["crs"]:
        raise ValueError(f"{name}: sistema de coordenadas diferente.")

    if info["shape"] != reference["shape"]:
        raise ValueError(f"{name}: dimensiones diferentes.")

    if not info["transform"].almost_equals(reference["transform"]):
        raise ValueError(f"{name}: alineación o resolución diferente.")

print("Las tres grillas coinciden espacialmente.")
print(f"Sistema de coordenadas: {reference['crs']}")
print(f"Resolución: {reference['res']}")
print(f"Unidad: {reference['crs'].linear_units}")

## 5. Comprobación de los outlets

Revisamos que cada outlet esté dentro del DEM y tenga datos válidos.
Esta comprobación no ajusta los puntos a la red de drenaje.

In [ ]:
# ── Revisar los datos en cada outlet ──────────────────────────────────────
outlet_checks = []

with (
    rasterio.open(DEM_PATH) as dem_src,
    rasterio.open(AREA_PATH) as area_src,
    rasterio.open(FD_PATH) as fd_src,
):
    for name, (x, y) in sample_outlets.items():
        row, col = dem_src.index(x, y)

        inside = (
            0 <= row < dem_src.height
            and 0 <= col < dem_src.width
        )

        z = np.nan
        drainage_area = np.nan
        status = "Fuera del DEM"

        if inside:
            values = [
                next(src.sample([(x, y)], masked=True))[0]
                for src in (dem_src, area_src, fd_src)
            ]

            valid = all(
                not np.ma.is_masked(value) and np.isfinite(value)
                for value in values
            )

            if valid:
                z = float(values[0])
                drainage_area = float(values[1])

                if drainage_area <= 0:
                    status = "Área no positiva"
                elif drainage_area < AREA_THRESHOLD:
                    status = "Área inferior al umbral"
                else:
                    status = "OK"
            else:
                status = "Datos ausentes o inválidos"

        outlet_checks.append({
            "rio": name,
            "fila": row,
            "columna": col,
            "elevacion_m": z,
            "area_valor_grilla": drainage_area,
            "estado": status,
        })

outlet_qc = pd.DataFrame(outlet_checks)
# display(outlet_qc #Borrar comentario si queiro ver el QC

n_ok = outlet_qc["estado"].eq("OK").sum()
print(f"Outlets sin observaciones: {n_ok}/{len(outlet_qc)}")

## 6. Selección de ríos

Seleccionamos por ID los ríos que queremos analizar y graficar.

In [ ]:
# ── Correspondencia entre ID y nombre ─────────────────────────────────────
# Conserva el orden de los 18 outlets definidos anteriormente.
river_ids = {
    str(i): name
    for i, name in enumerate(sample_outlets, start=1)
}

RIOS_MOSTRAR = list(river_ids)

# RIOS_MOSTRAR = ["1", "3", "16"]   # Mostrar estos tres ríos
# RIOS_MOSTRAR = ["6"]             # Mostrar solamente Reloca
# RIOS_MOSTRAR = list(river_ids)   # Mostrar los 18 ríos

selected_ids = list(dict.fromkeys(str(i) for i in RIOS_MOSTRAR))

invalid_ids = [
    river_id for river_id in selected_ids
    if river_id not in river_ids
]

if invalid_ids:
    raise ValueError(f"IDs no reconocidos: {invalid_ids}")

if not selected_ids:
    raise ValueError("Selecciona al menos un río.")

selected_outlets = {
    river_ids[river_id]: sample_outlets[river_ids[river_id]]
    for river_id in selected_ids
}

for river_id in selected_ids:
    print(f"{river_id}: {river_ids[river_id]}")

## 7. Ubicación de los outlets seleccionados

Mostramos los puntos sobre el DEM para revisar su ubicación.

In [ ]:
from rasterio.enums import Resampling
from rasterio.plot import plotting_extent

with rasterio.open(DEM_PATH) as src:
    factor = max(1, int(np.ceil(max(src.height, src.width) / 1500)))

    dem_preview = src.read(
        1,
        out_shape=(
            max(1, src.height // factor),
            max(1, src.width // factor),
        ),
        masked=True,
        resampling=Resampling.nearest,
    )
    extent = plotting_extent(src)

# ── Mapa de los outlets seleccionados ────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 8))

image = ax.imshow(
    dem_preview,
    extent=extent,
    origin="upper",
    cmap="terrain",
)

for river_id in selected_ids:
    name = river_ids[river_id]
    x, y = selected_outlets[name]

    ax.scatter(
        x, y,
        s=65,
        color=colors[name],
        edgecolor="black",
        linewidth=0.7,
        label=f"{river_id}: {name}",
        zorder=3,
    )

    ax.annotate(
        river_id,
        (x, y),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold",
        zorder=4,
    )

fig.colorbar(image, ax=ax, label="Elevación (m)", shrink=0.7)

ax.set(
    xlabel="Este (m)",
    ylabel="Norte (m)",
    title="Outlets seleccionados",
)
ax.set_aspect("equal")
ax.ticklabel_format(style="plain", useOffset=False)
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

## 8. Extracción del cauce principal

Recorremos el río aguas arriba, siguiendo el afluente de mayor área.
La distancia comienza en la celda del outlet.

In [ ]:
# ── Función para extraer un perfil longitudinal ──────────────────────────
def extraer_perfil(outlet, area_minima=AREA_THRESHOLD):
    row, col = fd._xy_to_rowscols((outlet,))[0]

    if row is None or col is None:
        raise ValueError("El outlet está fuera de la grilla.")

    nrows, ncols = area_grid.shape
    recorrido = []
    visitadas = set()

    while True:
        if (row, col) in visitadas:
            raise ValueError("Se detectó un ciclo en la dirección de flujo.")

        visitadas.add((row, col))
        drainage_area = float(area_grid[row, col])

        if not np.isfinite(drainage_area) or drainage_area < area_minima:
            if not recorrido:
                raise ValueError(
                    "El outlet no tiene área válida o está bajo el umbral."
                )
            break

        recorrido.append((row, col))

        # Vecinos que realmente drenan hacia la celda actual
        candidatos = []

        for r, c in fd.get_upstream_cell_indexes(row, col):
            r, c = int(r), int(c)

            if 0 <= r < nrows and 0 <= c < ncols:
                a = float(area_grid[r, c])

                if np.isfinite(a) and a >= area_minima:
                    candidatos.append((r, c))

        if not candidatos:
            break

        # Criterio de cauce principal: mayor área aportante
        row, col = max(
            candidatos,
            key=lambda rc: float(area_grid[rc[0], rc[1]])
        )

    if len(recorrido) < 2:
        raise ValueError(
            "El perfil tiene menos de dos puntos. Revisa el outlet y el umbral."
        )

    rows, cols = np.array(recorrido, dtype=int).T
    xy = np.asarray(fd._rowscols_to_xy(recorrido), dtype=float)
    x, y = xy[:, 0], xy[:, 1]

    # Distancia real entre centros de celdas, incluidas las diagonales
    pasos = np.hypot(np.diff(x), np.diff(y))
    distancia = np.concatenate(([0.0], np.cumsum(pasos)))

    perfil = pd.DataFrame({
        "fila": rows,
        "columna": cols,
        "x_utm": x,
        "y_utm": y,
        "distancia_m": distancia,
        "elevacion_m": elev_grid[rows, cols],
        "area_m2": area_grid[rows, cols],
    })

    # Conservar las posiciones sin elevación para no unir huecos en el gráfico
    perfil["elevacion_valida"] = np.isfinite(perfil["elevacion_m"])
    perfil.loc[~perfil["elevacion_valida"], "elevacion_m"] = np.nan

    # Diferencia entre la coordenada indicada y el centro de su celda
    perfil.attrs["ajuste_a_celda_m"] = float(
        np.hypot(x[0] - outlet[0], y[0] - outlet[1])
    )

    return perfil

## 9. Perfiles longitudinales

Mostramos elevación frente a distancia para los ríos seleccionados.

In [ ]:
# ── MODIFICAR AQUÍ: gráficos que quieres mostrar ──────────────────────────
RIOS_MOSTRAR = [1]

# RIOS_MOSTRAR = ["1", "3", "16"]   # Mostrar tres ríos
# RIOS_MOSTRAR = ["6"]             # Mostrar solo Reloca
# RIOS_MOSTRAR = list(river_ids)   # Mostrar los 18


# ── Validar los IDs ──────────────────────────────────────────────────────
selected_ids = list(dict.fromkeys(str(i) for i in RIOS_MOSTRAR))

if not selected_ids:
    raise ValueError("Selecciona al menos un río.")

invalid_ids = [i for i in selected_ids if i not in river_ids]

if invalid_ids:
    raise ValueError(f"IDs no reconocidos: {invalid_ids}")


# ── Extraer y graficar los perfiles seleccionados ─────────────────────────
perfiles = {}
resumen_perfiles = []

for river_id in selected_ids:
    name = river_ids[river_id]

    try:
        perfil = extraer_perfil(sample_outlets[name])
    except ValueError as error:
        print(f"{river_id} — {name}: {error}")
        continue

    perfiles[river_id] = perfil

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.plot(
        perfil["distancia_m"] / 1000,
        perfil["elevacion_m"],
        color=colors[name],
        linewidth=1.5,
    )

    ax.set(
        xlabel="Distancia desde el outlet (km)",
        ylabel="Elevación (m)",
        title=f"{river_id} — {name}",
    )
    ax.set_xlim(left=0)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    resumen_perfiles.append({
        "ID": river_id,
        "rio": name,
        "puntos": len(perfil),
        "longitud_km": perfil["distancia_m"].iloc[-1] / 1000,
        "puntos_sin_elevacion": int(
            (~perfil["elevacion_valida"]).sum()
        ),
        "ajuste_a_celda_m": perfil.attrs["ajuste_a_celda_m"],
    })

display(pd.DataFrame(resumen_perfiles))

#Para ver todos juntos descomentar
#plt.figure(figsize=(12, 7)); 
#[plt.plot((p := extraer_perfil(sample_outlets[n]))["distancia_m"] / 1000, 
#p["elevacion_m"], color=colors[n], label=f"{i}: {n}") for i, n in river_ids.items()]; 
#plt.xlabel("Distancia desde el outlet (km)"); plt.ylabel("Elevación (m)"); 
#plt.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left"); 
#plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 10. Cálculo de χ

Calculamos χ desde cada outlet y extraemos sus valores sobre el cauce principal.

In [ ]:
RIOS_CALCULAR = list(river_ids)


ids_calculo = list(dict.fromkeys(str(i) for i in RIOS_CALCULAR))

if not ids_calculo or any(i not in river_ids for i in ids_calculo):
    raise ValueError("Selecciona IDs válidos entre 1 y 18.")

# Se reinicia este diccionario cada vez que ejecutas la celda
perfiles_chi = {}

for river_id in ids_calculo:
    name = river_ids[river_id]
    outlet = sample_outlets[name]

    # Extraer nuevamente para utilizar el umbral actual
    perfil = extraer_perfil(outlet, area_minima=AREA_THRESHOLD)

    # Calcular χ con TopoAnalysis
    chi = d.Chi(
        flow_direction=fd,
        area=area,
        theta=THETA_REF,
        Ao=A0,
        outlets=[outlet],
    )

    rows = perfil["fila"].to_numpy(dtype=int)
    cols = perfil["columna"].to_numpy(dtype=int)
    chi_values = np.asarray(chi._griddata[rows, cols], dtype=float)

    if not np.all(np.isfinite(chi_values)):
        raise ValueError(f"{name}: hay valores de χ no válidos.")

    if not np.isclose(chi_values[0], 0.0):
        raise ValueError(f"{name}: χ no comienza en cero en el outlet.")

    if np.any(np.diff(chi_values) <= 0):
        raise ValueError(f"{name}: χ no aumenta continuamente aguas arriba.")

    perfil["chi_m"] = chi_values

    # Registrar los parámetros utilizados
    perfil.attrs.update({
        "theta_ref": THETA_REF,
        "A0_m2": A0,
        "area_minima_m2": AREA_THRESHOLD,
    })

    perfiles_chi[river_id] = perfil
    del chi   # Liberar la grilla completa antes del siguiente río

    print(f"{river_id} — {name}: χ calculado en {len(perfil):,} puntos.")

## 11. Perfiles χ–elevación

Mostramos los perfiles de los ríos seleccionados.

In [ ]:
RIOS_MOSTRAR = ["6"]

# RIOS_MOSTRAR = list(perfiles_chi)   # Todos los calculados

SUPERPONER = False

ids_grafico = list(dict.fromkeys(str(i) for i in RIOS_MOSTRAR))
faltantes = [i for i in ids_grafico if i not in perfiles_chi]

if not ids_grafico:
    raise ValueError("Selecciona al menos un río.")

if faltantes:
    raise ValueError(
        f"Primero calcula estos IDs en la celda 22: {faltantes}"
    )


def dibujar_chi(ax, river_id):
    name = river_ids[river_id]
    perfil = perfiles_chi[river_id]

    ax.plot(
        perfil["chi_m"],
        perfil["elevacion_m"],
        color=colors[name],
        linewidth=1.5,
        label=f"{river_id}: {name}",
    )

    ax.set(
        xlabel="χ (m)",
        ylabel="Elevación (m)",
    )
    ax.set_xlim(left=0)
    ax.grid(True, alpha=0.3)


if SUPERPONER:
    fig, ax = plt.subplots(figsize=(11, 6))

    for river_id in ids_grafico:
        dibujar_chi(ax, river_id)

    ax.set_title("Perfiles χ–elevación")
    ax.legend(
        fontsize=8,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    plt.tight_layout()
    plt.show()
    plt.close(fig)

else:
    for river_id in ids_grafico:
        fig, ax = plt.subplots(figsize=(9, 5))
        dibujar_chi(ax, river_id)
        ax.set_title(f"{river_id} — {river_ids[river_id]}")

        plt.tight_layout()
        plt.show()
        plt.close(fig)

## 12. Cálculo de ksn local

Estimamos la pendiente mediante una regresión local de elevación y distancia.

$$k_{sn}=S\,A^{\theta_{\mathrm{ref}}}$$

Conservamos las pendientes no positivas como control de calidad.

In [ ]:
# ── MODIFICAR AQUÍ: ventana para estimar la pendiente ─────────────────────
VENTANA_M = 500.0   # Longitud total de la ventana; valor inicial ajustable
MIN_PUNTOS = 5     # Mínimo de puntos válidos por regresión

# La ventana está centrada en cada punto: ± VENTANA_M / 2.
# Cerca de los extremos se utiliza una ventana parcial y se marca en la tabla.

if VENTANA_M <= 0 or MIN_PUNTOS < 3:
    raise ValueError("Usa una ventana positiva y al menos 3 puntos.")

if not perfiles_chi:
    raise ValueError("Primero calcula los perfiles en la celda 22.")

perfiles_ksn = {}

for river_id, perfil in perfiles_chi.items():
    resultado = perfil.copy()

    distance = resultado["distancia_m"].to_numpy()
    z = resultado["elevacion_m"].to_numpy()
    drainage_area = resultado["area_m2"].to_numpy()
    theta = perfil.attrs["theta_ref"]

    slope = np.full(len(resultado), np.nan)
    r2 = np.full(len(resultado), np.nan)
    n_points = np.zeros(len(resultado), dtype=int)
    half_window = VENTANA_M / 2

    for j, center in enumerate(distance):
        # Límites de la ventana a lo largo del cauce
        start = np.searchsorted(distance, center - half_window)
        end = np.searchsorted(
            distance, center + half_window, side="right"
        )

        x_window = distance[start:end]
        z_window = z[start:end]
        valid = np.isfinite(z_window)

        n_points[j] = valid.sum()

        if n_points[j] >= MIN_PUNTOS:
            fit = linregress(
                x_window[valid] - center,
                z_window[valid],
            )
            slope[j] = fit.slope
            r2[j] = fit.rvalue ** 2

    # La distancia aumenta aguas arriba: una pendiente normal es positiva.
    valid_ksn = (
        np.isfinite(slope)
        & (slope > 0)
        & np.isfinite(drainage_area)
        & (drainage_area > 0)
    )

    ksn = np.full(len(resultado), np.nan)
    ksn[valid_ksn] = (
        slope[valid_ksn] * drainage_area[valid_ksn] ** theta
    )

    resultado["pendiente"] = slope
    resultado["r2_pendiente"] = r2
    resultado["n_puntos_ventana"] = n_points
    resultado["ventana_parcial"] = (
        (distance - half_window < distance[0])
        | (distance + half_window > distance[-1])
    )
    resultado["pendiente_no_positiva"] = (
        np.isfinite(slope) & (slope <= 0)
    )
    resultado["ksn"] = ksn
    resultado["ksn_valido"] = valid_ksn

    resultado.attrs.update(perfil.attrs)
    resultado.attrs["ventana_m"] = VENTANA_M
    resultado.attrs["min_puntos"] = MIN_PUNTOS

    perfiles_ksn[river_id] = resultado

    print(
        f"{river_id} — {river_ids[river_id]}: "
        f"{valid_ksn.sum():,}/{len(resultado):,} valores válidos de ksn"
    )

## 13. Revisión de ksn

Mostramos ksn y resumimos las pendientes no positivas y las ventanas parciales.

In [ ]:
RIOS_REVISAR = ["6"] # = list(perfiles_ksn)   # Todos los calculados

# False: gráficos separados. True: perfiles superpuestos.
SUPERPONER = False

ids_revision = list(dict.fromkeys(str(i) for i in RIOS_REVISAR))

if not ids_revision:
    raise ValueError("Selecciona al menos un río.")

faltantes = [i for i in ids_revision if i not in perfiles_ksn]
if faltantes:
    raise ValueError(f"Primero calcula χ y ksn para estos IDs: {faltantes}")

# Comprobar que los perfiles superpuestos usen la misma concavidad
thetas = {perfiles_ksn[i].attrs["theta_ref"] for i in ids_revision}
if len(thetas) != 1:
    raise ValueError("Los perfiles tienen distintos valores de theta.")

theta_revision = next(iter(thetas))
unidad_ksn = f"m^{2 * theta_revision:g}"


def dibujar_ksn(ax, river_id):
    p = perfiles_ksn[river_id]
    name = river_ids[river_id]

    ax.plot(
        p["distancia_m"] / 1000,
        p["ksn"],
        color=colors[name],
        label=f"{river_id}: {name}",
        linewidth=1.2,
    )

    ax.set(
        xlabel="Distancia desde el outlet (km)",
        ylabel=f"ksn ({unidad_ksn})",
    )
    ax.set_xlim(left=0)
    ax.grid(alpha=0.3)


if SUPERPONER:
    fig, ax = plt.subplots(figsize=(11, 5))
    for river_id in ids_revision:
        dibujar_ksn(ax, river_id)

    ax.set_title("ksn de los ríos seleccionados")
    ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

else:
    for river_id in ids_revision:
        fig, ax = plt.subplots(figsize=(9, 5))
        dibujar_ksn(ax, river_id)
        ax.set_title(f"{river_id} — {river_ids[river_id]}")
        plt.tight_layout()
        plt.show()
        plt.close(fig)

# ── Tabla de control de calidad ──────────────────────────────────────────
resumen_qc = pd.DataFrame([
    {
        "ID": i,
        "rio": river_ids[i],
        "puntos": len(perfiles_ksn[i]),
        "ksn_validos": int(perfiles_ksn[i]["ksn_valido"].sum()),
        "pendientes_no_positivas": int(
            perfiles_ksn[i]["pendiente_no_positiva"].sum()
        ),
        "ventanas_parciales": int(
            perfiles_ksn[i]["ventana_parcial"].sum()
        ),
        "puntos_sin_pendiente": int(
            perfiles_ksn[i]["pendiente"].isna().sum()
        ),
        "ajuste_a_celda_m": perfiles_ksn[i].attrs["ajuste_a_celda_m"],
    }
    for i in ids_revision
])

display(resumen_qc)

## 14. Revisión espacial de los cauces

Mostramos cada cauce sobre el DEM junto con su outlet.]

In [ ]:
from rasterio.windows import Window, from_bounds

# ── MODIFICAR AQUÍ: margen alrededor del cauce ────────────────────────────
MARGEN_MAPA_M = 2000

# Usa los IDs elegidos en RIOS_REVISAR, celda 28.
for river_id in ids_revision:
    p = perfiles_ksn[river_id]
    name = river_ids[river_id]
    outlet = sample_outlets[name]

    with rasterio.open(DEM_PATH) as src:
        window = from_bounds(
            p["x_utm"].min() - MARGEN_MAPA_M,
            p["y_utm"].min() - MARGEN_MAPA_M,
            p["x_utm"].max() + MARGEN_MAPA_M,
            p["y_utm"].max() + MARGEN_MAPA_M,
            transform=src.transform,
        )

        # Ajustar a píxeles completos y a los límites del DEM
        col0 = max(0, int(np.floor(window.col_off)))
        row0 = max(0, int(np.floor(window.row_off)))
        col1 = min(src.width, int(np.ceil(window.col_off + window.width)))
        row1 = min(src.height, int(np.ceil(window.row_off + window.height)))
        window = Window(col0, row0, col1 - col0, row1 - row0)

        factor = max(1, int(np.ceil(max(window.width, window.height) / 1200)))

        preview = src.read(
            1,
            window=window,
            out_shape=(
                max(1, int(window.height) // factor),
                max(1, int(window.width) // factor),
            ),
            masked=True,
            resampling=Resampling.nearest,
        )

        left, bottom, right, top = src.window_bounds(window)

    fig, ax = plt.subplots(figsize=(8, 7))

    image = ax.imshow(
        preview,
        extent=(left, right, bottom, top),
        origin="upper",
        cmap="terrain",
    )

    ax.plot(
        p["x_utm"], p["y_utm"],
        color="blue", linewidth=1.2,
        label="Cauce extraído",
    )
    ax.scatter(
        *outlet, marker="*", s=130,
        color="red", edgecolor="black",
        label="Outlet indicado", zorder=4,
    )

    fig.colorbar(image, ax=ax, label="Elevación (m)", shrink=0.8)
    ax.set(
        title=f"{river_id} — {name}",
        xlabel="Este (m)",
        ylabel="Norte (m)",
    )
    ax.set_aspect("equal")
    ax.ticklabel_format(style="plain", useOffset=False)
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 15. Sensibilidad a la ventana de pendiente

Comparamos ksn con distintas ventanas, sin cambiar el resultado principal.

VENTANA_M define qué longitud del río usamos para estimar la pendiente en cada punto, que luego entra en el cálculo de ksn.
Por ejemplo, con VENTANA_M = 500, el código toma los puntos situados aproximadamente 250 m aguas abajo y 250 m aguas arriba, ajusta una recta a sus elevaciones y utiliza su pendiente.
- Ventana pequeña: muestra cambios cortos, pero también irregularidades o ruido del DEM.
- Ventana grande: produce un perfil de ksn más suave, pero puede atenuar cambios reales y estrechos, como un knickpoint.
Los 500 m son una elección inicial, no un valor obligatorio ni validado para tus ríos. La comparación entre 250, 500 y 1000 m permite ver cuánto depende el resultado de esa elección.

In [ ]:
# ── MODIFICAR AQUÍ: ventanas que quieres comparar ────────────────────────
VENTANAS_COMPARAR_M = [250, 500, 1000]

if not VENTANAS_COMPARAR_M or any(w <= 0 for w in VENTANAS_COMPARAR_M):
    raise ValueError("Las ventanas deben ser positivas.")

sensibilidad_ksn = {}
resumen_sensibilidad = []

for river_id in ids_revision:
    p = perfiles_ksn[river_id]

    distance = p["distancia_m"].to_numpy()
    z = p["elevacion_m"].to_numpy()
    area_values = p["area_m2"].to_numpy()
    theta = p.attrs["theta_ref"]
    min_points = p.attrs["min_puntos"]

    fig, ax = plt.subplots(figsize=(10, 5))

    for window_m in dict.fromkeys(VENTANAS_COMPARAR_M):
        slope = np.full(len(p), np.nan)

        for j, center in enumerate(distance):
            start = np.searchsorted(distance, center - window_m / 2)
            end = np.searchsorted(
                distance, center + window_m / 2, side="right"
            )

            xw = distance[start:end]
            zw = z[start:end]
            valid = np.isfinite(zw)

            if valid.sum() >= min_points:
                slope[j] = linregress(
                    xw[valid] - center, zw[valid]
                ).slope

        valid_ksn = (
            np.isfinite(slope) & (slope > 0)
            & np.isfinite(area_values) & (area_values > 0)
        )

        ksn_values = np.full(len(p), np.nan)
        ksn_values[valid_ksn] = (
            slope[valid_ksn] * area_values[valid_ksn] ** theta
        )

        partial = (
            (distance - window_m / 2 < distance[0])
            | (distance + window_m / 2 > distance[-1])
        )

        # Guardar también pendientes rechazadas y ventanas parciales
        sensibilidad_ksn[(river_id, window_m)] = pd.DataFrame({
            "distancia_m": distance,
            "pendiente": slope,
            "ksn": ksn_values,
            "ksn_valido": valid_ksn,
            "ventana_parcial": partial,
        })

        ax.plot(
            distance / 1000,
            ksn_values,
            label=f"Ventana de {window_m:g} m",
            linewidth=1.2,
        )

        interior = valid_ksn & ~partial
        resumen_sensibilidad.append({
            "ID": river_id,
            "ventana_m": window_m,
            "puntos_validos_interiores": int(interior.sum()),
            "mediana_ksn_interior": (
                float(np.median(ksn_values[interior]))
                if interior.any() else np.nan
            ),
        })

    ax.set(
        title=f"{river_id} — {river_ids[river_id]}",
        xlabel="Distancia desde el outlet (km)",
        ylabel=f"ksn (m^{2 * theta:g})",
    )
    ax.set_xlim(left=0)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
    plt.close(fig)

resumen_sensibilidad = pd.DataFrame(resumen_sensibilidad)
display(resumen_sensibilidad)

## Visualización

Resumen por río y comparación de perfiles con origen común o desplazado.
Los desplazamientos son visuales y no modifican los resultados.

In [ ]:
RIOS_VER = ["1"]    # =list(perfiles_ksn)     # Todos los ríos calculados

# Desplazamientos horizontales para la vista desfasada.
# Un valor positivo mueve el perfil hacia la derecha.
# Los IDs no incluidos tendrán desplazamiento cero.
# Distancia adicional desde una referencia común hasta cada outlet:
DESFASES_KM = {
    # "1": 0.0,
    # "3": 2.5,    # EJEMPLO: reemplazar por la distancia que corresponda
    # "16": 5.0,
}

# Completa esto solo si tienes un desfase de χ definido.
DESFASES_CHI_M = {
    # "1": 0.0,
    # "3": 100.0,  # EJEMPLO: reemplazar por el valor que corresponda
}

MOSTRAR_EXTREMOS_KSN = True

GROSOR = 1.4

# PREPARACIÓN

ids_ver = list(dict.fromkeys(str(i) for i in RIOS_VER))

if not ids_ver:
    raise ValueError("Selecciona al menos un río.")

faltantes = [i for i in ids_ver if i not in perfiles_ksn]
if faltantes:
    raise ValueError(f"No hay resultados de ksn para estos IDs: {faltantes}")

desfases_km = {str(i): float(v) for i, v in DESFASES_KM.items()}
desfases_chi = {str(i): float(v) for i, v in DESFASES_CHI_M.items()}

if not all(np.isfinite(v) for v in [*desfases_km.values(), *desfases_chi.values()]):
    raise ValueError("Los desplazamientos deben ser números finitos.")


def datos_visuales(river_id, desfasado=False):
    """Preparar copias de los valores; no modificar las tablas originales."""
    p = perfiles_ksn[river_id]

    distancia = p["distancia_m"].to_numpy(copy=True) / 1000
    chi = p["chi_m"].to_numpy(copy=True)
    elevacion = p["elevacion_m"].to_numpy(copy=True)
    ksn = p["ksn"].to_numpy(copy=True)

    if not MOSTRAR_EXTREMOS_KSN:
        ksn[p["ventana_parcial"].to_numpy(dtype=bool)] = np.nan

    if desfasado:
        distancia += desfases_km.get(river_id, 0.0)
        chi += desfases_chi.get(river_id, 0.0)

    return distancia, chi, elevacion, ksn


def trazar_tres(axs, river_id, desfasado=False):
    name = river_ids[river_id]
    distancia, chi, elevacion, ksn = datos_visuales(
        river_id, desfasado
    )

    estilo = {
        "color": colors[name],
        "linewidth": GROSOR,
        "label": f"{river_id}: {name}",
    }

    axs[0].plot(distancia, elevacion, **estilo)
    axs[1].plot(chi, elevacion, **estilo)
    axs[2].plot(distancia, ksn, **estilo)


def dar_formato(axs, theta, desfasado=False):
    etiqueta_distancia = (
        "Distancia + desfase (km)"
        if desfasado else "Distancia desde el outlet (km)"
    )
    etiqueta_chi = "χ + desfase (m)" if desfasado else "χ (m)"

    axs[0].set(
        title="Perfil longitudinal",
        xlabel=etiqueta_distancia,
        ylabel="Elevación (m)",
    )
    axs[1].set(
        title="Perfil χ–elevación",
        xlabel=etiqueta_chi,
        ylabel="Elevación (m)",
    )
    axs[2].set(
        title="ksn",
        xlabel=etiqueta_distancia,
        ylabel=f"ksn (m^{2 * theta:g})",
    )

    for ax in axs:
        ax.grid(alpha=0.3)
        if not desfasado:
            ax.set_xlim(left=0)

# VISTAS DISPONIBLES

def ver_por_rio():
    """Una figura de tres paneles por cada río seleccionado."""
    for river_id in ids_ver:
        p = perfiles_ksn[river_id]
        fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))

        trazar_tres(axs, river_id)
        dar_formato(axs, p.attrs["theta_ref"])

        fig.suptitle(
            f"{river_id} — {river_ids[river_id]} | "
            f"θ={p.attrs['theta_ref']} | "
            f"Ventana={p.attrs['ventana_m']:g} m"
        )
        plt.tight_layout()
        plt.show()
        plt.close(fig)


def ver_superpuestos(desfasado=False):
    """Una figura de tres paneles con todos los ríos seleccionados."""
    # Comparar perfiles calculados con los mismos parámetros
    for parametro in ("theta_ref", "A0_m2", "ventana_m", "area_minima_m2"):
        valores = {
            perfiles_ksn[i].attrs[parametro]
            for i in ids_ver
        }
        if len(valores) != 1:
            raise ValueError(
                f"Los ríos tienen distintos valores de {parametro}. "
                "Unifica los parámetros antes de superponerlos."
            )

    p = perfiles_ksn[ids_ver[0]]
    fig, axs = plt.subplots(1, 3, figsize=(16, 5))

    for river_id in ids_ver:
        trazar_tres(axs, river_id, desfasado)

    dar_formato(axs, p.attrs["theta_ref"], desfasado)

    if desfasado:
        titulo = "Perfiles superpuestos con desplazamientos horizontales"

        if not any(desfases_chi.get(i, 0) != 0 for i in ids_ver):
            axs[1].set_title("Perfil χ–elevación: sin desfase de χ")
    else:
        titulo = "Perfiles superpuestos: distancia y χ = 0 en cada outlet"

    fig.suptitle(titulo)

    handles, labels = axs[0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc="upper left",
        bbox_to_anchor=(1.0, 0.9),
        fontsize=9,
    )

    plt.tight_layout()
    plt.show()
    plt.close(fig)

# MOSTRAR: COMENTA CON # LAS LÍNEAS QUE NO QUIERAS EJECUTAR

ver_por_rio()                         # Tres gráficos por río

#ver_superpuestos(desfasado=False)     # Outlets en un origen horizontal común

#ver_superpuestos(desfasado=True)    # Activar después de definir los desfases

## Guardar Graficos

Guardamos los gráficos individuales y una figura conjunta
en una carpeta con el nombre de cada río seleccionado.

In [ ]:
from pathlib import Path
from datetime import datetime
import re

# ════════════════════════════════════════════════════════════════════════
# MODIFICAR AQUÍ
# ════════════════════════════════════════════════════════════════════════

RIOS_GUARDAR = ["6"]

# RIOS_GUARDAR = ["1", "3", "16"]
# RIOS_GUARDAR = list(perfiles_ksn)   # Todos los calculados

CARPETA_BASE = Path(
    r"C:\datos\Memoria"
    r"\Codigos Stanford\Productos\figuras"
)

FORMATOS = ["png", "pdf"]
DPI = 300
INCLUIR_EXTREMOS_KSN = True


# ── Validar selección ────────────────────────────────────────────────────
ids_guardar = list(dict.fromkeys(str(i) for i in RIOS_GUARDAR))

if not ids_guardar:
    raise ValueError("Selecciona al menos un río.")

faltantes = [i for i in ids_guardar if i not in perfiles_ksn]
if faltantes:
    raise ValueError(f"No hay resultados para estos IDs: {faltantes}")

formatos = list(dict.fromkeys(f.lower().lstrip(".") for f in FORMATOS))

if not formatos or any(f not in ("png", "pdf") for f in formatos):
    raise ValueError("Usa png, pdf o ambos.")

# Fecha en los archivos para conservar las exportaciones anteriores
fecha = datetime.now().strftime("%Y%m%d_%H%M%S_%f")


def guardar(fig, carpeta, nombre):
    for formato in formatos:
        fig.savefig(
            carpeta / f"{nombre}_{fecha}.{formato}",
            dpi=DPI,
            bbox_inches="tight",
            facecolor="white",
        )


# ── Generar y guardar gráficos ───────────────────────────────────────────
for river_id in ids_guardar:
    p = perfiles_ksn[river_id]
    name = river_ids[river_id]
    theta = p.attrs["theta_ref"]
    ventana = p.attrs["ventana_m"]

    # Ejemplo: "Rio Reloca" → "Rio_Reloca"
    nombre_carpeta = re.sub(r'[<>:"/\\|?*]', "_", name)
    nombre_carpeta = re.sub(r"\s+", "_", nombre_carpeta).strip(" .")

    carpeta_rio = CARPETA_BASE / nombre_carpeta
    carpeta_rio.mkdir(parents=True, exist_ok=True)

    distancia = p["distancia_m"].to_numpy() / 1000
    ksn = p["ksn"].to_numpy(copy=True)

    if not INCLUIR_EXTREMOS_KSN:
        ksn[p["ventana_parcial"].to_numpy(dtype=bool)] = np.nan

    paneles = [
        {
            "archivo": "longitudinal",
            "titulo": "Perfil longitudinal",
            "x": distancia,
            "y": p["elevacion_m"],
            "xlabel": "Distancia desde el outlet (km)",
            "ylabel": "Elevación (m)",
        },
        {
            "archivo": "chi",
            "titulo": "Perfil χ–elevación",
            "x": p["chi_m"],
            "y": p["elevacion_m"],
            "xlabel": "χ (m)",
            "ylabel": "Elevación (m)",
        },
        {
            "archivo": "ksn",
            "titulo": "ksn",
            "x": distancia,
            "y": ksn,
            "xlabel": "Distancia desde el outlet (km)",
            "ylabel": f"ksn (m^{2 * theta:g})",
        },
    ]

    def dibujar_panel(ax, panel):
        ax.plot(
            panel["x"],
            panel["y"],
            color=colors[name],
            linewidth=1.4,
        )
        ax.set(
            title=panel["titulo"],
            xlabel=panel["xlabel"],
            ylabel=panel["ylabel"],
        )
        ax.set_xlim(left=0)
        ax.grid(alpha=0.3)

    # ── Tres figuras individuales ────────────────────────────────────────
    for panel in paneles:
        fig, ax = plt.subplots(figsize=(9, 5))

        try:
            dibujar_panel(ax, panel)
            fig.suptitle(f"{river_id} — {name}")

            if panel["archivo"] == "chi":
                ax.set_title(
                    f"Perfil χ–elevación | θ={theta} | "
                    f"A₀={p.attrs['A0_m2']:g} m²"
                )
            elif panel["archivo"] == "ksn":
                ax.set_title(f"ksn | θ={theta} | Ventana={ventana:g} m")

            fig.tight_layout()
            guardar(fig, carpeta_rio, panel["archivo"])

        finally:
            plt.close(fig)

    # ── Una figura con los tres paneles juntos ────────────────────────────
    fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))

    try:
        for ax, panel in zip(axs, paneles):
            dibujar_panel(ax, panel)

        fig.suptitle(
            f"{river_id} — {name} | "
            f"θ={theta} | A₀={p.attrs['A0_m2']:g} m² | "
            f"Ventana={ventana:g} m"
        )
        fig.tight_layout()
        guardar(fig, carpeta_rio, "longitudinal_chi_ksn")

    finally:
        plt.close(fig)

    print(f"{name}: {4 * len(formatos)} archivos guardados.")
    print(f"Carpeta: {carpeta_rio}\n")



## Exportar perfiles

Guardamos las tablas y los parámetros de todos los ríos calculados.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np

BASE = Path(
    r"C:\datos\Memoria"
    r"\Codigos Stanford\Productos"
)

if not perfiles_ksn:
    raise ValueError("Primero calcula los perfiles del notebook anterior.")

carpeta = Path(
    r"C:\datos\Memoria"
    r"\Codigos Stanford\Productos\info_perfiles\20260929" #Modificar la fecha cada vez que corra el codigo
)

# Evita sobrescribir una exportación existente.
carpeta.mkdir(parents=True, exist_ok=False)

manifest = {
    "criterio_cauce": "Mayor area aportante en cada confluencia",
    "unidad_distancia": "m",
    "unidad_area": "m2",
    "rios": {},
}

for river_id, perfil in perfiles_ksn.items():
    river_id = str(river_id)
    archivo = f"rio_{int(river_id):02d}.csv"

    perfil.to_csv(
        carpeta / archivo,
        index=False,
        encoding="utf-8-sig",
    )

    manifest["rios"][river_id] = {
        "nombre": river_ids[river_id],
        "archivo": archivo,
        "outlet_xy": list(sample_outlets[river_ids[river_id]]),
        "parametros": dict(perfil.attrs),
    }


def convertir_numpy(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f"Tipo no exportable: {type(value)}")


(carpeta / "manifest.json").write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        default=convertir_numpy,
    ),
    encoding="utf-8",
)

print(f"Ríos exportados: {len(manifest['rios'])}")
print(f"Carpeta de entrada para el nuevo notebook:\n{carpeta}")

In [ ]:
# ── Exportar red completa y cauces principales de los outlets ──────────
from pathlib import Path
from datetime import datetime
from collections import Counter
import numpy as np
import rasterio
from osgeo import ogr, osr

# CONFIGURACIÓN
RUTA_REDES = Path(
    r"C:\datos\Memoria"
    r"\Codigos Stanford\Productos\Redes_de_drenaje"
)

AREA_MIN_RED = float(AREA_THRESHOLD)  # m²; actualmente 1 km²
RIOS_EXPORTAR = list(river_ids)      # Los 18 outlets
# RIOS_EXPORTAR = ["6", "10"]        # Alternativa: solo estos ríos

# ── Validaciones ──────────────────────────────────────────────────────
if not np.isfinite(AREA_MIN_RED) or AREA_MIN_RED <= 0:
    raise ValueError("AREA_MIN_RED debe ser positiva y finita.")

ids_red = list(dict.fromkeys(str(i) for i in RIOS_EXPORTAR))
if not ids_red or any(i not in river_ids for i in ids_red):
    raise ValueError("Selecciona IDs válidos en RIOS_EXPORTAR.")

with rasterio.open(AREA_PATH) as src:
    crs_red = src.crs
    mascara_red = (
        (src.read_masks(1) > 0)
        & np.isfinite(area_grid)
        & (area_grid >= AREA_MIN_RED)
    )

if crs_red is None or crs_red.to_epsg() != 32718:
    raise ValueError("Se esperaba WGS 84 / UTM 18S, EPSG:32718.")

nfilas, ncolumnas = area_grid.shape

# Extraer en variables nuevas: no reemplazar perfiles, χ ni ksn.
principales_red = {}
for rid in ids_red:
    nombre = river_ids[rid]
    principales_red[rid] = extraer_perfil(
        sample_outlets[nombre],
        area_minima=AREA_MIN_RED,
    )

# ── Construir las conexiones de la red ───────────────────────────────
# Se usa la función de TopoAnalysis del propio notebook.
# No se presupone una codificación D8 externa.
celdas_red = np.flatnonzero(mascara_red.ravel())
receptor_red = {}  # celda aguas arriba -> celda aguas abajo

print(
    f"Construyendo red: {len(celdas_red):,} celdas "
    f"con área ≥ {AREA_MIN_RED / 1e6:g} km²..."
)

for posicion, celda in enumerate(celdas_red, start=1):
    celda = int(celda)
    fila, columna = divmod(celda, ncolumnas)

    for r, c in fd.get_upstream_cell_indexes(fila, columna):
        r, c = int(r), int(c)

        if (
            0 <= r < nfilas
            and 0 <= c < ncolumnas
            and mascara_red[r, c]
        ):
            aguas_arriba = r * ncolumnas + c

            if (
                aguas_arriba in receptor_red
                and receptor_red[aguas_arriba] != celda
            ):
                raise ValueError("Una celda tiene dos receptores D8.")

            receptor_red[aguas_arriba] = celda

    if posicion % 100000 == 0:
        print(f"  Revisadas {posicion:,} celdas.")

if not receptor_red:
    raise ValueError("No se encontraron conexiones con este umbral.")

# Una línea por tramo entre cabeceras, confluencias y terminaciones.
numero_afluentes = Counter(receptor_red.values())
inicios_red = [
    celda for celda in receptor_red
    if numero_afluentes[celda] != 1
]

# ── Preparar salida y sistema de coordenadas ─────────────────────────
carpeta_red = RUTA_REDES / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
carpeta_red.mkdir(parents=True, exist_ok=False)

srs_red = osr.SpatialReference()
srs_red.ImportFromWkt(crs_red.to_wkt())
srs_red.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)

ogr.UseExceptions()
driver_red = ogr.GetDriverByName("ESRI Shapefile")

if driver_red is None:
    raise RuntimeError("GDAL no tiene disponible el controlador Shapefile.")


def crear_shp_red(ruta, campos):
    ds = driver_red.CreateDataSource(str(ruta))
    capa = ds.CreateLayer(
        ruta.stem,
        srs=srs_red,
        geom_type=ogr.wkbLineString,
        options=["ENCODING=UTF-8"],
    )

    for nombre, tipo in campos:
        campo = ogr.FieldDefn(nombre, tipo)
        if tipo == ogr.OFTString:
            campo.SetWidth(100)
        capa.CreateField(campo)

    return ds, capa


def agregar_linea_red(capa, xy, atributos):
    xy = np.asarray(xy, dtype=float)

    if len(xy) < 2 or not np.isfinite(xy).all():
        raise ValueError("Geometría con menos de dos puntos o coordenadas inválidas.")

    geometria = ogr.Geometry(ogr.wkbLineString)
    for x, y in xy:
        geometria.AddPoint_2D(float(x), float(y))

    registro = ogr.Feature(capa.GetLayerDefn())
    registro.SetGeometry(geometria)

    for nombre, valor in atributos.items():
        registro.SetField(nombre, valor)

    capa.CreateFeature(registro)
    registro = None


# ── 1. Red completa del DEM ──────────────────────────────────────────
ruta_completa = carpeta_red / "red_completa.shp"

ds_red, capa_red = crear_shp_red(
    ruta_completa,
    [
        ("tramo_id", ogr.OFTInteger),
        ("long_m", ogr.OFTReal),
        ("amin_km2", ogr.OFTReal),
    ],
)

enlaces_exportados = set()
ntramos_red = 0

try:
    for inicio in inicios_red:
        recorrido = [inicio]
        actual = inicio

        while actual in receptor_red:
            if actual in enlaces_exportados:
                raise ValueError("Conexión repetida o ciclo en la red.")

            enlaces_exportados.add(actual)
            siguiente = receptor_red[actual]
            recorrido.append(siguiente)

            if numero_afluentes[siguiente] != 1:
                break

            actual = siguiente

        filas_columnas = [
            divmod(celda, ncolumnas) for celda in recorrido
        ]
        xy = np.asarray(
            fd._rowscols_to_xy(filas_columnas), dtype=float
        )
        longitud = float(np.hypot(
            np.diff(xy[:, 0]), np.diff(xy[:, 1])
        ).sum())

        ntramos_red += 1
        agregar_linea_red(
            capa_red,
            xy,
            {
                "tramo_id": ntramos_red,
                "long_m": longitud,
                "amin_km2": AREA_MIN_RED / 1e6,
            },
        )

    if len(enlaces_exportados) != len(receptor_red):
        raise ValueError(
            "Quedaron conexiones sin exportar. Posible ciclo D8; "
            "no utilices esta exportación hasta revisarlo."
        )
finally:
    capa_red = None
    ds_red = None

# ── 2. Cauces principales asociados a tus outlets ────────────────────
ruta_outlets = carpeta_red / "rios_outlets.shp"

ds_red, capa_red = crear_shp_red(
    ruta_outlets,
    [
        ("rio_id", ogr.OFTString),
        ("rio", ogr.OFTString),
        ("long_m", ogr.OFTReal),
        ("amin_km2", ogr.OFTReal),
        ("ajuste_m", ogr.OFTReal),
    ],
)

try:
    for rid, perfil_red in principales_red.items():
        # El perfil original comienza en el outlet.
        # Invertimos solo la geometría exportada para orientarla aguas abajo.
        xy = perfil_red[["x_utm", "y_utm"]].to_numpy(dtype=float)[::-1]

        agregar_linea_red(
            capa_red,
            xy,
            {
                "rio_id": rid,
                "rio": river_ids[rid],
                "long_m": float(perfil_red["distancia_m"].iloc[-1]),
                "amin_km2": AREA_MIN_RED / 1e6,
                "ajuste_m": float(
                    perfil_red.attrs["ajuste_a_celda_m"]
                ),
            },
        )
finally:
    capa_red = None
    ds_red = None

# ── Verificar que ambos archivos puedan abrirse ──────────────────────
for ruta, cantidad in (
    (ruta_completa, ntramos_red),
    (ruta_outlets, len(principales_red)),
):
    ds_revision = ogr.Open(str(ruta))
    if ds_revision is None:
        raise RuntimeError(f"No se pudo abrir el resultado: {ruta}")

    capa_revision = ds_revision.GetLayer()
    if capa_revision.GetFeatureCount() != cantidad:
        raise RuntimeError(f"Número de geometrías incorrecto: {ruta}")

    capa_revision = None
    ds_revision = None

print("\nExportación terminada.")
print(f"Red completa: {ntramos_red:,} tramos.")
print(f"Cauces principales: {len(principales_red)} ríos.")
print(f"Umbral: {AREA_MIN_RED / 1e6:g} km².")
print("CRS: EPSG:32718.")
print(f"\nCarpeta:\n{carpeta_red}")